# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrasannaSaiS/machinelearning01-flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Type: Ranking / Scoring** — not classification, not clustering. Every low/zero-AI page gets a continuous **GEO Priority Score**, sorted into a queue. The `framing-ml-problems` skill's own mapping table says it plainly: *"Which ones first?" → Ranking / scoring* — and "which pages first" is exactly the decision I named in ML-02.

**Why not classification.** `docs/ml-intern-dataset-and-lane-guide.md` is explicit about this exact freestyle lane: *"Do not train a binary classifier on AI sessions alone; with positives this rare, the model will look impressive and mean nothing."* The code below shows why in one number: only 6.43% of pages have any AI-referred session at all, so a classifier that always predicts "no" scores ~93.6% accuracy while catching **zero** real opportunities. That's not a model, that's a coin flip in a lab coat. (Classification isn't wrong everywhere in this repo — `is_declining_label` is a legitimate classification target elsewhere, built from an *observed* future trend window. It's specifically wrong **here**, for **this** signal, because of how sparse and thin it is.)

**Why not clustering.** FlyRank's own research (`docs/flyrank-seo-research-march-2026.pdf`, ML Appendix) already ran a 5-segment k-means over this exact portfolio. Clustering answers *"what kinds of pages exist"* — useful, but it hands back segments, not an **order**. It can't tell an editor which of two pages in the same cluster to fix first, and "which first" is the whole decision.

**Why ranking/scoring wins.** It matches the decision, the action (a ranked review queue), and the lane guide's own sanctioned use for this exact freestyle direction: *"EDA; broad pattern analysis; opportunity ranking."*

In [2]:
import os, subprocess
import pandas as pd
import numpy as np

def load_starter_csv():
    """Portable loader: works from a local repo clone or a fresh Colab session."""
    local_candidates = [
        "../../data/raw/content_refresh_anonymized.csv",
    ]
    for path in local_candidates:
        if os.path.exists(path):
            return pd.read_csv(path)
    repo_dir = "machinelearning01-flyrank"
    if not os.path.isdir(repo_dir):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/PrasannaSaiS/machinelearning01-flyrank.git", repo_dir],
            check=True,
        )
    return pd.read_csv(f"{repo_dir}/data/raw/content_refresh_anonymized.csv")

df = load_starter_csv()
df["ai_sessions_90d"] = df["ai_sessions_90d"].fillna(0)
df["has_ai"] = (df["ai_sessions_90d"] > 0).astype(int)

total_pages = len(df)
positives = int(df["has_ai"].sum())
positive_rate = positives / total_pages

dumb_accuracy = 1 - positive_rate     # "always predict no AI referral"
dumb_recall = 0.0                     # it catches zero real opportunities

decision_surface = df[(df["has_ai"] == 0) & (df["impressions_90d"] >= 100)]

print("=" * 74)
print("WHY NOT CLASSIFICATION -- the trap, in numbers")
print("=" * 74)
print(f"Rows in starter slice                 : {total_pages:,}")
print(f"Rows with ai_sessions_90d > 0          : {positives:,} ({positive_rate*100:.2f}% positive rate)")
print(f"'Always predict no' accuracy           : {dumb_accuracy:.4f}")
print(f"'Always predict no' recall             : {dumb_recall:.2f}  <- catches 0 real opportunities")
print()
print("=" * 74)
print("WHY RANKING FITS -- the actual decision surface")
print("=" * 74)
print(f"Visible pages (>=100 impressions) with ZERO AI referral : {len(decision_surface):,}")
print(f"  ...as a share of the full slice                        : {len(decision_surface)/total_pages*100:.1f}%")
print("This is the queue a ranking model orders -- not a class label a classifier predicts.")

WHY NOT CLASSIFICATION -- the trap, in numbers
Rows in starter slice                 : 30,000
Rows with ai_sessions_90d > 0          : 1,930 (6.43% positive rate)
'Always predict no' accuracy           : 0.9357
'Always predict no' recall             : 0.00  <- catches 0 real opportunities

WHY RANKING FITS -- the actual decision surface
Visible pages (>=100 impressions) with ZERO AI referral : 20,207
  ...as a share of the full slice                        : 67.4%
This is the queue a ranking model orders -- not a class label a classifier predicts.


## 2. Target or proxy

**What I score:** not "will this page be cited" — I can't observe that. `docs/data-dictionary.md` is blunt about the one AI-specific column I have: `ai_sessions_90d` counts *"click-throughs from AI assistants, not citations or rankings."* A page can be cited and never clicked; this data would show nothing.

**Where the label comes from — a DEFINED proxy, said honestly, not an observed outcome.** The real, observed AI-signal (`ai_sessions_90d > 0`) exists but is too sparse to train on (per the lane guide's own warning in Section 1) and too shallow semantically (click, not citation). So I build a **GEO Priority Score** from features that are dense, always observed, and — this is the part I actually tested rather than assumed — *empirically shown to travel with the rare real signal I do have.*

Here's the reversal: my ML-02 filter leaned on `scroll_rate` (engagement depth) as the "obviously right" GEO lever. Checked against real correlations, it isn't. `scroll_rate` correlates with `ai_sessions_90d` at r = −0.04 and `engagement_rate` at r = +0.02 — statistically nothing. `word_count` (r = +0.21), `impressions_90d` (r = +0.22), and `sessions_90d` (r = +0.31) all correlate far more. FlyRank's own portfolio-scale research finds the identical shape: AI-attracting pages differ mainly in **higher impressions and heavier word count** (the 5,000+ word bucket has by far the highest AI-page rate), not in engagement depth. I'm revising ML-02's instinct in light of that, not defending it.

**Proxy definition:** `geo_priority_score = 0.5 × percentile(log(impressions_90d)) + 0.5 × percentile(word_count)`. I deliberately exclude `scroll_rate` / `engagement_rate` (tested — doesn't help) and `trend_direction` / `trend_pct` (the label trap — never a feature, per the data dictionary).

In [3]:
import os, subprocess
import pandas as pd
import numpy as np

def load_starter_csv():
    """Portable loader: works from a local repo clone or a fresh Colab session."""
    local_candidates = [
        "data/raw/content_refresh_anonymized.csv",
        "../../data/raw/content_refresh_anonymized.csv",
        "../../../data/raw/content_refresh_anonymized.csv",
    ]
    for path in local_candidates:
        if os.path.exists(path):
            return pd.read_csv(path)
    repo_dir = "machinelearning01-flyrank"
    if not os.path.isdir(repo_dir):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/PrasannaSaiS/machinelearning01-flyrank.git", repo_dir],
            check=True,
        )
    return pd.read_csv(f"{repo_dir}/data/raw/content_refresh_anonymized.csv")

df = load_starter_csv()
df["ai_sessions_90d"] = df["ai_sessions_90d"].fillna(0)
df["has_ai"] = (df["ai_sessions_90d"] > 0).astype(int)

def percentile_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

# Reality check before defining the proxy: does the "obvious" engagement
# signal actually track AI referral? Test it, don't assume it.
print("Correlation with ai_sessions_90d (the one real AI-specific signal I have):")
for col in ["scroll_rate", "engagement_rate", "word_count", "impressions_90d", "sessions_90d"]:
    sub = df[[col, "ai_sessions_90d"]].dropna()
    r = sub[col].corr(sub["ai_sessions_90d"])
    print(f"  {col:18s} r = {r:+.3f}")

print()
print("FlyRank's own portfolio research (docs/flyrank-seo-research-march-2026.pdf) finds the")
print("same shape at much larger scale: AI-attracting pages differ mainly in higher impressions")
print("and heavier word count (highest AI-page rate sits in the 5,000+ word bucket) -- NOT in")
print("scroll or engagement depth. My ML-02 filter leaned on scroll_rate -- the data says that")
print("was the wrong lever.")

# The proxy target: a defined composite, built only from validated correlates
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["depth_score"] = percentile_rank(df["word_count"].fillna(df["word_count"].median()))
df["geo_priority_score"] = 0.5 * df["visibility_score"] + 0.5 * df["depth_score"]

print()
print("Top 5 pages by GEO Priority Score (a DEFINED proxy, not an observed outcome):")
cols_to_show = ["content_id", "client_id", "word_count", "impressions_90d", "has_ai", "geo_priority_score"]
print(df.sort_values("geo_priority_score", ascending=False)[cols_to_show].head(5).to_string(index=False))

Correlation with ai_sessions_90d (the one real AI-specific signal I have):
  scroll_rate        r = -0.044
  engagement_rate    r = +0.016
  word_count         r = +0.208
  impressions_90d    r = +0.217
  sessions_90d       r = +0.313

FlyRank's own portfolio research (docs/flyrank-seo-research-march-2026.pdf) finds the
same shape at much larger scale: AI-attracting pages differ mainly in higher impressions
and heavier word count (highest AI-page rate sits in the 5,000+ word bucket) -- NOT in
scroll or engagement depth. My ML-02 filter leaned on scroll_rate -- the data says that
was the wrong lever.

Top 5 pages by GEO Priority Score (a DEFINED proxy, not an observed outcome):
          content_id         client_id  word_count  impressions_90d  has_ai  geo_priority_score
content_2dba2b1f9536 client_6208ef0f77      7676.0           443434       1            0.997967
content_26e14d59e257 client_6208ef0f77      8417.0            93308       1            0.996400
content_a965a1fc5544 clien

## 3. Success metric

**Metric: Precision@50.** ML-02's action was *"a ranked review queue for content teams"* — so the number that matters isn't overall accuracy (meaningless here — 93.6% is just the base rate from Section 1) or a portfolio-wide statistic like ROC-AUC that no single editor experiences. It's: **of the top 50 pages the queue hands an editor this cycle, how many are actually the right 50?** The lane guide agrees — its thresholds section recommends exactly this: *"Precision@50 if the team can act on 50 candidates."*

**What "good" means, defended with a number:** the population base rate is 6.43%, so a random top-50 list contains ~3 relevant pages by chance. That's the floor. "Good" means a real, measurable lift over it — not a feeling.

**Honesty check I owe here:** because true positives are so sparse, Precision@K computed against `has_ai_sessions` is a *directional sense-check* on whether my proxy score points the right way — not a training objective. I'm using the rare real signal to validate, exactly as the lane guide scopes this direction ("EDA and ranking," not classification training).

In [4]:
import os, subprocess
import pandas as pd
import numpy as np

def load_starter_csv():
    """Portable loader: works from a local repo clone or a fresh Colab session."""
    local_candidates = [
        "data/raw/content_refresh_anonymized.csv",
        "../../data/raw/content_refresh_anonymized.csv",
        "../../../data/raw/content_refresh_anonymized.csv",
    ]
    for path in local_candidates:
        if os.path.exists(path):
            return pd.read_csv(path)
    repo_dir = "machinelearning01-flyrank"
    if not os.path.isdir(repo_dir):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/PrasannaSaiS/machinelearning01-flyrank.git", repo_dir],
            check=True,
        )
    return pd.read_csv(f"{repo_dir}/data/raw/content_refresh_anonymized.csv")

df = load_starter_csv()
df["ai_sessions_90d"] = df["ai_sessions_90d"].fillna(0)
df["has_ai"] = (df["ai_sessions_90d"] > 0).astype(int)

def percentile_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": y_true, "s": scores}).sort_values("s", ascending=False).head(k)
    return frame["y"].mean(), int(frame["y"].sum())

df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["depth_score"] = percentile_rank(df["word_count"].fillna(df["word_count"].median()))
df["scroll_score"] = percentile_rank(df["scroll_rate"].fillna(0))
df["geo_priority_score"] = 0.5 * df["visibility_score"] + 0.5 * df["depth_score"]

base_rate = df["has_ai"].mean()
k = 50
print(f"Editor capacity assumed (from ML-02's action): {k} pages/review cycle -> metric = Precision@{k}")
print(f"Random-draw floor at this K (population base rate) : {base_rate:.3f}  ({base_rate*100:.1f}%)")
print("Any score at/below this line is no better than a shuffled deck. Above it is real lift.\n")

print(f"{'Ranking rule':36s} {'Precision@'+str(k):>12s} {'hits/'+str(k):>10s} {'lift x base':>12s}")
for name, col in [
    ("scroll_rate only  (my ML-02 filter)", "scroll_score"),
    ("impressions only", "visibility_score"),
    ("word_count only", "depth_score"),
    ("GEO Priority Score (combined)", "geo_priority_score"),
]:
    p, hits = precision_at_k(df["has_ai"], df[col], k)
    print(f"{name:36s} {p:12.3f} {str(hits)+'/'+str(k):>10s} {p/base_rate:11.1f}x")

Editor capacity assumed (from ML-02's action): 50 pages/review cycle -> metric = Precision@50
Random-draw floor at this K (population base rate) : 0.064  (6.4%)
Any score at/below this line is no better than a shuffled deck. Above it is real lift.

Ranking rule                         Precision@50    hits/50  lift x base
scroll_rate only  (my ML-02 filter)         0.000       0/50         0.0x
impressions only                            0.360      18/50         5.6x
word_count only                             0.480      24/50         7.5x
GEO Priority Score (combined)               0.820      41/50        12.7x


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymous content page's trailing-90-day performance snapshot, for one pseudonymous client.** Not a query, not a click, not a daily time-series point — that finer grain (`fact_content_daily_performance`) lives in the warehouse for later weeks. `content_id` / `client_id` are pseudonyms for grouping and joins only, per the data dictionary — never features.

My lane's actual operating slice narrows this further to the rows the decision applies to: visible pages (`impressions_90d ≥ 100`) currently earning **zero** measured AI referral — the exact candidate pool a ranked queue would be built from.

In [5]:
import os, subprocess
import pandas as pd
import numpy as np

def load_starter_csv():
    """Portable loader: works from a local repo clone or a fresh Colab session."""
    local_candidates = [
        "data/raw/content_refresh_anonymized.csv",
        "../../data/raw/content_refresh_anonymized.csv",
        "../../../data/raw/content_refresh_anonymized.csv",
    ]
    for path in local_candidates:
        if os.path.exists(path):
            return pd.read_csv(path)
    repo_dir = "machinelearning01-flyrank"
    if not os.path.isdir(repo_dir):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/PrasannaSaiS/machinelearning01-flyrank.git", repo_dir],
            check=True,
        )
    return pd.read_csv(f"{repo_dir}/data/raw/content_refresh_anonymized.csv")

df = load_starter_csv()
df["ai_sessions_90d"] = df["ai_sessions_90d"].fillna(0)
print(f"Full starter slice: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("One row = one pseudonymous CONTENT PAGE, aggregated over its trailing 90-day window,")
print("for one pseudonymous CLIENT. Never a query, a click, or a daily time-series point --")
print("that finer grain lives in fact_content_daily_performance in the warehouse (later weeks).\n")

df["has_ai"] = (df["ai_sessions_90d"] > 0).astype(int)
lane_slice = df[(df["has_ai"] == 0) & (df["impressions_90d"] >= 100)].copy()
print(f"My lane's operating slice (the ranking queue's candidate pool): {lane_slice.shape[0]:,} rows")
print("Same 'one row = one page' grain -- just filtered to the pages the decision applies to:\n")

show_cols = ["content_id", "client_id", "content_type", "main_intent",
             "word_count", "impressions_90d", "avg_position", "ai_sessions_90d"]
print(lane_slice[show_cols].head(5).to_string(index=False))
print()
print(lane_slice[show_cols].dtypes)

Full starter slice: 30,000 rows x 44 columns
One row = one pseudonymous CONTENT PAGE, aggregated over its trailing 90-day window,
for one pseudonymous CLIENT. Never a query, a click, or a daily time-series point --
that finer grain lives in fact_content_daily_performance in the warehouse (later weeks).

My lane's operating slice (the ranking queue's candidate pool): 20,207 rows
Same 'one row = one page' grain -- just filtered to the pages the decision applies to:

          content_id         client_id    content_type   main_intent  word_count  impressions_90d  avg_position  ai_sessions_90d
content_304f48230142 client_f369cb89fc keyword article transactional      3221.0             3803          10.6                0
content_a1fb4e703a9e client_4e07408562 keyword article informational      2481.0            15320          20.3                0
content_9aa793d4d895 client_7f2253d7e2 keyword article informational      3515.0            12581          36.5                0
content_331d6c4

## 5. Why ML beats a fixed rule here

Two concrete failures rule out an if-statement:

**1. No single signal has a clean threshold.** Rank by `scroll_rate` alone (my ML-02 instinct) and Precision@50 is **0.000** — worse than doing nothing, because the "obviously right" GEO lever isn't the one the data supports. Rank by the *right* single signal (`word_count` alone) and you only reach 48% — capped, because the relationship isn't even monotonic: the AI-referral rate **drops** from 3.73% (<800 words) to 2.58% (800–1500 words) before climbing to 28.73% (5,000+ words). There's no `X` in `if word_count > X` that doesn't misclassify a whole bucket.

**2. The pattern doesn't transfer across categories.** `comparison article` — structurally the most "citation-ready" format on paper (explicit head-to-head structure) — has the *lowest* AI-referral rate of the three content types (0.86% vs 6.65% for keyword articles). A rule tuned on one `content_type` actively misleads on another.

**What a combination buys you that a rule can't:** combining `impressions` and `word_count` (my Section 2 proxy) reaches **82%** Precision@50 — not the average of the two single-signal scores (36% and 48%), *more than either* — a genuine interaction, not an if/elif chain summing two independent checks. Learning how signals like these trade off, non-linearly, across categories, is exactly the "real pattern, too messy for hand-written thresholds" case the `framing-ml-problems` skill describes.

In [6]:
import os, subprocess
import pandas as pd
import numpy as np

def load_starter_csv():
    """Portable loader: works from a local repo clone or a fresh Colab session."""
    local_candidates = [
        "data/raw/content_refresh_anonymized.csv",
        "../../data/raw/content_refresh_anonymized.csv",
        "../../../data/raw/content_refresh_anonymized.csv",
    ]
    for path in local_candidates:
        if os.path.exists(path):
            return pd.read_csv(path)
    repo_dir = "machinelearning01-flyrank"
    if not os.path.isdir(repo_dir):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/PrasannaSaiS/machinelearning01-flyrank.git", repo_dir],
            check=True,
        )
    return pd.read_csv(f"{repo_dir}/data/raw/content_refresh_anonymized.csv")

df = load_starter_csv()
df["ai_sessions_90d"] = df["ai_sessions_90d"].fillna(0)
df["has_ai"] = (df["ai_sessions_90d"] > 0).astype(int)

print("PROOF 1 -- a single word_count threshold has no clean cut point (non-monotonic):")
bins = [-1, 800, 1500, 3000, 5000, 100000]
labels = ["<800", "800-1500", "1500-3000", "3000-5000", "5000+"]
df["wc_bucket"] = pd.cut(df["word_count"], bins=bins, labels=labels)
bucket_table = df.groupby("wc_bucket", observed=True)["has_ai"].agg(rate="mean", n="count")
bucket_table["rate"] = (bucket_table["rate"] * 100).round(2)
print(bucket_table.to_string())
print("-> rate DROPS from <800 to 800-1500 words, then climbs. Any 'if word_count > X' rule")
print("   either misses the short-content signal or muddies the middle bucket. There is no X.\n")

print("PROOF 2 -- the same word-count logic doesn't transfer across content_type:")
type_table = df.groupby("content_type")["has_ai"].agg(rate="mean", n="count")
type_table["rate"] = (type_table["rate"] * 100).round(2)
print(type_table.sort_values("rate", ascending=False).to_string())
print("-> 'comparison article' is the most structurally citation-ready format on paper")
print("   (explicit head-to-head structure) yet has the LOWEST AI-referral rate of the three.")
print("   A rule tuned on keyword articles actively misleads on this type.")

PROOF 1 -- a single word_count threshold has no clean cut point (non-monotonic):
            rate     n
wc_bucket             
<800        3.73   322
800-1500    2.58  2906
1500-3000   4.00  9511
3000-5000   6.86  6910
5000+      28.73  2652
-> rate DROPS from <800 to 800-1500 words, then climbs. Any 'if word_count > X' rule
   either misses the short-content signal or muddies the middle bucket. There is no X.

PROOF 2 -- the same word-count logic doesn't transfer across content_type:
                    rate      n
content_type                   
keyword article     6.65  27207
feedly article      5.49   2096
comparison article  0.86    697
-> 'comparison article' is the most structurally citation-ready format on paper
   (explicit head-to-head structure) yet has the LOWEST AI-referral rate of the three.
   A rule tuned on keyword articles actively misleads on this type.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.